<table style="width: 100%; border-collapse: collapse; border: none; background: #fffbeb; border-left: 6px solid #f59e0b; border-radius: 8px; padding: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">  <tr style="border: none;">    <td style="vertical-align: middle; border: none; padding: 15px 20px;">      <h1 style="margin: 0; color: #78350f; font-size: 2em; font-family: system-ui, -apple-system, sans-serif; font-weight: 800; letter-spacing: -0.02em;">        🧮 01. Criterios de División: Gini y Entropía      </h1>      <p style="margin: 6px 0 0 0; color: #b45309; font-size: 1.15em; font-weight: 600; font-family: system-ui, -apple-system, sans-serif;">        Especialización en Ciencia de Datos | Programación para Ciencia de Datos      </p>      <p style="margin: 4px 0 0 0; color: #92400e; font-size: 0.95em; font-family: system-ui, -apple-system, sans-serif;">        Universidad Santo Tomás — Seccional Tunja      </p>    </td>    <td style="text-align: right; vertical-align: middle; border: none; padding: 15px 20px; width: 30%;">      <span style="background: #f59e0b; color: #ffffff; padding: 6px 14px; border-radius: 20px; font-size: 0.85em; font-weight: 700; display: inline-block; margin-bottom: 8px;">        💡 Para Dummies • Módulo 09      </span><br>      <span style="color: #78350f; font-size: 0.85em;">Docente: Santiago A. Zúñiga M.</span><br>      <a href="mailto:gestorvirtualcienciadatos@ustatunja.edu.co" style="color: #b45309; font-size: 0.8em; text-decoration: none; font-weight: 500;">gestorvirtualcienciadatos@ustatunja.edu.co</a>    </td>  </tr></table><div align="center" style="margin-top: 15px; margin-bottom: 15px;">  <a href="https://colab.research.google.com/github/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/blob/main/Data%20Science%20programming/09%20-%20Decision%20Trees/Para%20Dummies/01_Criterios_Division_y_Arboles_Clasificacion_Dummies.ipynb" target="_parent">    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="vertical-align: middle;"/>  </a></div>

---
## ¿Qué vamos a aprender aquí? 🎈

En el cuaderno anterior vimos **qué** es un árbol de decisión (raíz, nodos, hojas) y confiamos "a ciegas" en que Scikit-Learn elegía las mejores preguntas. Aquí abrimos esa caja negra: vamos a entender **cómo** decide el algoritmo cuál pregunta es la mejor en cada paso.

No necesitas saber logaritmos de memoria — solo entender una idea central: **qué tan "revuelto" está un grupo de datos**, y cómo el árbol busca dejarlo lo menos revuelto posible.

---
## 1. CART: un jugador de "veinte preguntas" que siempre elige la mejor pregunta 📐

El algoritmo que usa Scikit-Learn se llama **CART** (*Classification and Regression Trees*) y funciona de forma "voraz" (*greedy*): en cada nodo, prueba **todas** las preguntas posibles sobre **todas** las variables (`¿horas_estudio > 3?`, `¿horas_estudio > 4.5?`, `¿horas_sueno > 6?`...) y se queda con la que deja los dos grupos resultantes **más parecidos internamente** (más "puros").

Luego repite el mismo proceso, por separado, dentro de cada uno de esos dos grupos nuevos. Y así sucesivamente. "Voraz" significa que en cada paso elige lo mejor **para ese paso**, sin adivinar el futuro — como en "veinte preguntas", donde eliges la pregunta que más reduce las opciones *ahora mismo*, sin saber qué preguntas vendrán después.

---
## 2. ¿Qué tan "revuelta" está una bolsa de dulces? 🍬

Imagina dos bolsas de dulces:

- **Bolsa A:** 10 dulces, los 10 son de fresa. Muy **pura** — si metes la mano, sabes exactamente qué vas a sacar.
- **Bolsa B:** 10 dulces, 5 de fresa y 5 de limón. Muy **revuelta** — no tienes ni idea de cuál vas a sacar.

Esa "revoltura" es justo lo que miden matemáticamente el **Índice de Gini** y la **Entropía**. Ambos valen **0 cuando el grupo es 100% puro** (como la Bolsa A) y alcanzan su **valor máximo cuando el grupo está lo más mezclado posible** (50%-50%, como la Bolsa B).

| Criterio | Idea intuitiva | Fórmula (2 clases) |
|---|---|---|
| **Gini** | Probabilidad de "equivocarte" si asignas etiquetas al azar según las proporciones del grupo | $Gini = 1 - p_0^2 - p_1^2$ |
| **Entropía** | "Sorpresa" promedio (viene de la teoría de la información) | $H = -p_0 \log_2(p_0) - p_1 \log_2(p_1)$ |

En cada división, el árbol calcula qué tan "revuelto" queda cada grupo hijo y elige la pregunta que **más reduce la revoltura promedio** respecto al grupo padre. A esa reducción se le llama **Ganancia de Información**.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

np.random.seed(7)

print("Entorno listo para explorar los criterios de división Gini y Entropía.")

---
## 3. Creamos un mini-set de correos (para detectar spam) 📧

Vamos a construir, a mano, un grupito pequeño de "correos" ficticios con dos pistas muy simples:

- `tiene_gratis`: si el correo contiene la palabra "gratis" (1) o no (0).
- `num_signos`: cuántos signos de exclamación (`!`) tiene el asunto.

Y la etiqueta que queremos predecir: `es_spam` (1 = spam, 0 = correo legítimo). A propósito dejamos algunos correos "raros" (por ejemplo, uno legítimo con muchos signos) para que el grupo no quede perfectamente puro desde el principio.

In [ ]:
correos = pd.DataFrame({
    'tiene_gratis': [1, 1, 1, 0, 0, 0, 1, 0, 1, 0, 1, 0],
    'num_signos':   [3, 4, 1, 0, 0, 2, 5, 1, 0, 0, 2, 4],
    'es_spam':      [1, 1, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0]
})

correos

### 🤔 ¿Qué acaba de pasar?

Creamos una tabla de 12 correos ficticios con dos características (`tiene_gratis`, `num_signos`) y la etiqueta real (`es_spam`). Fíjate que la fila con `tiene_gratis=1` y `num_signos=0` **no** es spam: dejamos ese "caso raro" a propósito, para que el grupo no sea 100% predecible con una sola pista, tal como pasa con datos reales.

---
## 4. Entrenando árboles con Gini vs Entropía 🌲

Scikit-Learn nos deja elegir el criterio con el parámetro `criterion`. Vamos a entrenar dos árboles idénticos, uno con cada criterio, y comparar qué tan bien clasifican estos mismos correos.

In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree

X = correos[['tiene_gratis', 'num_signos']]
y = correos['es_spam']

arbol_gini = DecisionTreeClassifier(criterion='gini', max_depth=2, random_state=42)
arbol_gini.fit(X, y)

arbol_entropia = DecisionTreeClassifier(criterion='entropy', max_depth=2, random_state=42)
arbol_entropia.fit(X, y)

print(f"Exactitud con Gini:     {arbol_gini.score(X, y) * 100:.1f}%")
print(f"Exactitud con Entropía: {arbol_entropia.score(X, y) * 100:.1f}%")

### 🤔 ¿Qué acaba de pasar?

Entrenamos dos árboles con exactamente los mismos datos, cambiando solo el criterio de división. En la práctica, Gini y Entropía suelen elegir preguntas muy parecidas (a veces idénticas) y terminan con una exactitud similar — la diferencia entre ambos casi nunca es dramática. La razón para preferir uno u otro suele ser de **velocidad de cálculo** (Gini evita los logaritmos, así que es un poco más rápido de calcular) más que de precisión final.

In [ ]:
plt.figure(figsize=(10, 5), dpi=110)
plot_tree(
    arbol_gini,
    feature_names=['tiene_gratis', 'num_signos'],
    class_names=['Legítimo', 'Spam'],
    filled=True,
    rounded=True,
    fontsize=10
)
plt.title("Árbol de correos entrenado con criterio Gini", fontweight='bold')
plt.show()

### 🤔 ¿Qué acaba de pasar?

En la caja de la raíz puedes leer el valor de `gini` **antes** de dividir: eso mide qué tan revuelto estaba el grupo completo de 12 correos. A medida que bajas por el árbol, fíjate cómo el valor de `gini` de cada caja va bajando — cada pregunta deja los grupos hijos más "puros" que el grupo padre. Cuando `gini` llega a `0.0`, esa hoja quedó perfectamente pura (todos los correos de esa caja son de la misma clase).

---
## 5. Calculando la Entropía a mano, paso a paso 🧮

Para que la fórmula deje de sentirse abstracta, vamos a calcularla nosotros mismos sobre el grupo completo de 12 correos (7 spam, 5 legítimos), sin usar Scikit-Learn — solo para comprobar que el número que arroja el árbol tiene sentido.

In [ ]:
def entropia(positivos, total):
    # Entropía de un grupo con 'positivos' casos de la clase 1, de un total 'total'
    if positivos == 0 or positivos == total:
        return 0.0
    p1 = positivos / total
    p0 = 1 - p1
    return -(p0 * np.log2(p0) + p1 * np.log2(p1))

# Grupo completo: 7 correos spam de 12 en total
H_padre = entropia(7, 12)
print(f"Entropía del grupo completo (padre): {H_padre:.3f}")
print("Como no es ni 0 (puro) ni 1 (máxima mezcla en 2 clases), confirma que el grupo está parcialmente revuelto.")

### 🤔 ¿Qué acaba de pasar?

- La función `entropia` implementa exactamente la fórmula $H = -p_0 \log_2(p_0) - p_1 \log_2(p_1)$ que vimos antes.
- Le pasamos "7 correos spam de 12 en total" y obtenemos un número entre 0 y 1: mientras más cerca de 1, más revuelto (mezclado 50%-50%) está el grupo; mientras más cerca de 0, más puro.
- Este es exactamente el mismo cálculo que Scikit-Learn hace internamente (miles de veces) para decidir, en cada nodo, cuál pregunta reduce más esta "revoltura" — eso es la **Ganancia de Información**.

---
## 6. Resumen relámpago ⚡

| Idea | En una frase |
|---|---|
| CART | Algoritmo "voraz": en cada nodo prueba todas las preguntas posibles y elige la mejor **para ese paso**. |
| Impureza | Qué tan mezclado está un grupo respecto a sus clases (0 = puro, máximo = 50%-50% en 2 clases). |
| Índice de Gini | Mide la probabilidad de "equivocarte" si etiquetas al azar según las proporciones del grupo. |
| Entropía | Mide la "sorpresa" promedio del grupo (viene de la teoría de la información). |
| Ganancia de Información | Cuánto se reduce la impureza promedio de los hijos respecto al padre, gracias a una pregunta. |
| `criterion='gini'` / `'entropy'` | El parámetro de `DecisionTreeClassifier` para elegir cuál medida de impureza usar. |

➡️ **Siguiente paso:** en el cuaderno [02 - Árboles de Regresión y Poda Cost-Complexity (Para Dummies)](02_Arboles_Regresion_y_Poda_Cost_Complexity_Dummies.ipynb) veremos qué pasa cuando lo que queremos predecir **no** es una categoría sino un número (como un precio), y cómo evitar que el árbol se "aprenda de memoria" los datos de entrenamiento.

---<div align="center">  <p style="font-size: 0.9em; color: #64748b;">    © 2026 <b>Universidad Santo Tomás — Seccional Tunja</b><br>    <i>Especialización en Ciencia de Datos | Programación para Ciencia de Datos (Edición Para No Ingenieros)</i>  </p></div>